# 05 — Data Collection for Imitation Learning

Record robot trajectories and export for training imitation learning models.

**What you'll learn**:
- Use `TrajectoryRecorder` to capture observation-action pairs
- Record multiple episodes with metadata
- Save to HDF5 format (NumPy/PyTorch compatible)
- Export to LeRobot format (HuggingFace Hub compatible)

**Prerequisites**: `pip install threewe[sim,data]`

In [ ]:
import asyncio
import numpy as np
from threewe import Robot
from threewe.data.recorder import TrajectoryRecorder

In [ ]:
robot = Robot(backend="gazebo", scene="office_v2")
robot.connect()

## Set Up the Recorder

The `TrajectoryRecorder` captures images, poses, velocities, and actions at each step.
Optionally record LiDAR and depth data too.

In [ ]:
recorder = TrajectoryRecorder(
    robot,
    record_lidar=True,   # include 360° LiDAR scan
    record_depth=False,  # skip depth to save memory
)

## Record an Episode

An "episode" is one demonstration trajectory.
In practice, you'd teleoperate the robot; here we simulate with a scripted policy.

In [ ]:
# Episode 1: Drive forward with slight left turn
recorder.start_episode(metadata={
    "task": "navigate_to_desk",
    "demonstrator": "scripted_policy",
})

for step in range(20):
    vx = 0.2
    vy = 0.0
    omega = 0.1 * np.sin(step * 0.3)  # gentle oscillation

    robot.set_velocity(vx, vy, omega)
    await asyncio.sleep(0.1)
    recorder.record_step(action=[vx, vy, omega])

robot.stop()
ep1 = recorder.end_episode()
print(f"Episode 1: {ep1.length} steps, {ep1.duration:.1f}s")

In [ ]:
# Episode 2: Random exploration
recorder.start_episode(metadata={
    "task": "random_exploration",
    "demonstrator": "random_policy",
})

for step in range(30):
    vx = np.random.uniform(0.0, 0.3)
    vy = np.random.uniform(-0.1, 0.1)
    omega = np.random.uniform(-0.5, 0.5)

    robot.set_velocity(vx, vy, omega)
    await asyncio.sleep(0.1)
    recorder.record_step(action=[vx, vy, omega])

robot.stop()
ep2 = recorder.end_episode()
print(f"Episode 2: {ep2.length} steps, {ep2.duration:.1f}s")

In [ ]:
print(f"\nRecorder summary:")
print(f"  Episodes: {recorder.num_episodes}")
print(f"  Total steps: {recorder.total_steps}")

## Save to HDF5

HDF5 is the standard format for NumPy/PyTorch data loading.
The structure is:
```
/episode_000/
    images: (T, H, W, 3) uint8
    poses: (T, 3) float32
    velocities: (T, 3) float32
    actions: (T, 3) float32
    timestamps: (T,) float64
    lidar: (T, N) float32  [if recorded]
```

In [ ]:
output_path = "data/demo_trajectories.h5"
recorder.save_hdf5(output_path)
print(f"Saved to {output_path}")

## Load and Inspect HDF5 Data

In [ ]:
try:
    import h5py

    with h5py.File(output_path, "r") as f:
        print(f"HDF5 file contents:")
        print(f"  Num episodes: {f.attrs['num_episodes']}")
        print(f"  Total steps: {f.attrs['total_steps']}")
        print()

        ep0 = f["episode_000"]
        print(f"  episode_000:")
        for key in ep0.keys():
            print(f"    {key}: shape={ep0[key].shape}, dtype={ep0[key].dtype}")

except ImportError:
    print("Install h5py to inspect: pip install h5py")

## Export to LeRobot Format

LeRobot uses Parquet (tabular data) + MP4 (video) — the standard for
HuggingFace Hub. This format is compatible with training VLA models
like Pi0, SmolVLA, and ACT.

In [ ]:
lerobot_dir = "data/lerobot_export"
recorder.save_lerobot(lerobot_dir)
print(f"Exported to {lerobot_dir}/")
print("  data/     — Parquet files (poses, velocities, actions)")
print("  videos/   — MP4 video per episode")
print("  meta/     — Dataset metadata (info.json)")

## Next Steps

With recorded data, you can:
1. Train an imitation learning model (ACT, Diffusion Policy)
2. Upload to HuggingFace Hub for sharing
3. Deploy the trained policy back to the robot:

```python
from threewe.ai.vla_runner import VLARunner

policy = VLARunner.from_local("checkpoints/my_policy")

async with Robot(backend="real") as robot:
    while not done:
        obs = robot.get_observation()
        action = policy.predict(obs, instruction="navigate to desk")
        robot.execute_action(action)
```

In [ ]:
robot.disconnect()
print("Done!")